In [1]:
import os
if os.path.basename(os.getcwd()) != "notebooks":
    os.chdir("/home/jovyan/notebooks")
print("cwd:", os.getcwd())

cwd: /home/jovyan/notebooks


---
> **Transparence sur l'usage d'un assistant IA**
>
> Plusieurs cellules de ce notebook (indiquées individuellement par un
> commentaire à la première personne, ex. *"J'ai adapté..."*, *"J'ai corrigé..."*)
> ont été complétées avec l'aide de l'assistant IA **Claude / Claude Code**
> (Anthropic), après que nous avons constaté que certaines cellules d'exercice
> reposaient sur des données ou variables absentes du projet réel (schéma
> différent du jeu de données réellement fourni, plage de dates 2022-2023 de
> l'énoncé alors que nos données couvrent 2020-2021, fichiers "pré-préparés"
> jamais générés par aucune cellule).
>
> Chaque correction a été vérifiée par rapport aux données réelles (schéma,
> plage de dates, comptages) avant d'être acceptée -- voir la section
> *« Transparence sur l'utilisation d'un assistant IA »* du `README.md` du
> projet pour le détail complet de la démarche et de ce qui reste de notre
> propre analyse (choix de conception, interprétation des résultats).
---


In [2]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, BooleanType, TimestampType
)
import time

# Je repars de la même config que ce matin (Session 3) : mêmes chemins,
# même app Spark avec Delta activé.
# ── Chemins ─────────────────────────────────────────────────────────────────
DATA_DIR           = Path("../data")
OUTPUT_DIR         = DATA_DIR / "output"
VELIB_CONSOLIDE    = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE   = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES      = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR  = OUTPUT_DIR / "stream_input"
STREAM_CHECKPOINT  = OUTPUT_DIR / "checkpoints"

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Paramètres ───────────────────────────────────────────────────────────────
APP_NAME      = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED          = 42

# ── SparkSession avec Delta ─────────────────────────────────────────────────
# J'ai enlevé .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") :
# ça déclenchait une résolution Ivy/Maven au démarrage qui plantait le JVM
# (FileNotFoundException sur /home/jovyan/.ivy2/cache, volume Docker mal
# permissionné) -- et c'était de toute façon inutile : le JAR Delta est déjà
# installé en dur dans l'image (/usr/local/spark/jars/), comme en Session 3,
# qui n'a jamais eu besoin de cette ligne. J'aligne aussi driver.memory sur 4g
# comme ce matin, pour rester cohérent et limiter le risque de OOM.
spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession active : {spark.version}")
print(f"Spark UI : http://localhost:4040")
print(f"STREAM_SOURCE_DIR : {STREAM_SOURCE_DIR}")
print(f"Nombre de fichiers déjà dans STREAM_SOURCE_DIR : {len(list(STREAM_SOURCE_DIR.glob('*.json')))}")

SparkSession active : 3.5.3
Spark UI : http://localhost:4040
STREAM_SOURCE_DIR : ../data/output/stream_input
Nombre de fichiers déjà dans STREAM_SOURCE_DIR : 2


---
# PARTIE 2 -- Structured Streaming (après-midi)

## 2.1 Concepts fondamentaux

### Du batch au streaming

Le batch traite un ensemble de données **figé** : on sait à l'avance combien
il y a de lignes, on peut trier, on peut faire des jointures complètes.

Le streaming traite un flux de données **continu** : les données arrivent
en permanence, on ne connaît pas la fin, et certains événements peuvent
arriver en retard.

Spark Structured Streaming répond à ce problème avec un modèle élégant :
le flux est traité comme une **table infinie** qui s'agrandit au fil du temps.
Les requêtes SQL ou DataFrame s'y appliquent sans modification.

```
Flux d'entrée  :  [evt1] [evt2] [evt3] ...  [evtN]  ...
                     |      |      |              |
                  +--+------+------+--------------+----→ temps
                  |
                  Spark : exécution de micro-batchs toutes les X secondes
                  |
                  Résultat : table de résultats mise à jour en continu
```

### Vocabulaire

| Terme | Définition |
|-------|-----------|
| **Trigger** | Fréquence d'exécution des micro-batchs |
| **Checkpoint** | Sauvegarde de l'état pour la reprise après panne |
| **Watermark** | Délai maximal toléré pour les données tardives |
| **Fenêtre glissante** | Agrégation sur une plage de temps mobile |
| **Sink** | Destination de l'écriture (console, fichier, Delta...) |


---
## 2.2 Le simulateur de flux

En production, le flux proviendrait de Kafka ou de l'API GBFS en direct.
Pour ce cours, un script Python **rejoue les données historiques** en écrivant
des fichiers JSON dans un répertoire surveillé par Spark.

Ce mécanisme -- la **file source** (file source) -- est le moyen le plus simple
de tester Structured Streaming sans infrastructure externe.

> Le simulateur est fourni dans `scripts/simulateur_flux.py`.
> Lancez-le dans un terminal séparé **avant** d'exécuter les cellules suivantes.
>
> ```bash
> python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3
> ```
>
> L'option `--vitesse 3` signifie que chaque seconde réelle correspond à
> 3 minutes de données historiques.


In [3]:
# Vérification : le simulateur tourne-t-il ?
# (avant de lancer cette cellule, j'ai démarré dans un terminal :
#  python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3)
import time, os

def compter_fichiers_stream(attente_sec: int = 5) -> int:
    """Attend et compte les fichiers JSON produits par le simulateur."""
    time.sleep(attente_sec)
    fichiers = list(STREAM_SOURCE_DIR.glob("*.json"))
    return len(fichiers)

n = compter_fichiers_stream(3)
if n == 0:
    print("[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.")
    print(f"  Vérifiez que le simulateur tourne dans un terminal séparé.")
    print(f"  Répertoire surveillé : {STREAM_SOURCE_DIR}")
else:
    print(f"[OK] {n} fichiers JSON présents dans {STREAM_SOURCE_DIR}")
    # Aperçu du dernier fichier produit
    fichiers = sorted(STREAM_SOURCE_DIR.glob("*.json"))
    dernier = fichiers[-1]
    print(f"\nDernier fichier : {dernier.name}")
    print("Contenu (2 premières lignes) :")
    with open(dernier) as f:
        for i, line in enumerate(f):
            if i >= 2:
                break
            print(f"  {line.strip()[:200]}")

[OK] 2 fichiers JSON présents dans ../data/output/stream_input

Dernier fichier : tardif_3min_1788201664.json
Contenu (2 premières lignes) :
  [{"nom_station": "Station Test Watermark", "capacite": 20, "velos_meca": 0, "velos_elec": 0, "velos_total": 0, "horodatage": "2026-08-31T18:38:04Z"}, {"nom_station": "Station Test Watermark", "capacit


---
## 2.3 Source de streaming : `readStream`

`spark.readStream` fonctionne exactement comme `spark.read`, avec deux différences :

1. Il retourne un **DataFrame de streaming** (pas un DataFrame ordinaire).
2. Il ne peut pas être affiché directement avec `.show()` -- il faut déclencher
   une **requête de streaming** avec `.writeStream`.


In [4]:
# Schéma du flux JSON produit par le simulateur
# J'ai adapté ce schéma en regardant scripts/simulateur_flux.py : il n'émet
# ni station_id, ni code_arr, ni bornettes_libres -- seulement nom_station,
# capacite, velos_meca, velos_elec, velos_total et horodatage. Comme au
# Jour 2 matin, j'identifie donc chaque station par son nom, et je recalculerai
# bornettes_libres moi-même dans la cellule suivante.
schema_flux = StructType([
    StructField("nom_station",  StringType(),    True),
    StructField("capacite",     IntegerType(),   True),
    StructField("velos_meca",   IntegerType(),   True),
    StructField("velos_elec",   IntegerType(),   True),
    StructField("velos_total",  IntegerType(),   True),
    StructField("horodatage",   TimestampType(), True),
])

# Création du DataFrame de streaming
# Chaque fichier contient un tableau JSON de plusieurs snapshots (voir
# json.dump(payload, ...) dans le simulateur) -> il faut multiLine=true,
# sinon Spark s'attend à un objet JSON par ligne et rejette tout le fichier.
# maxFilesPerTrigger=2 : au plus 2 fichiers par micro-batch.
stream_df = (
    spark.readStream
    .format("json")
    .schema(schema_flux)
    .option("multiLine", "true")
    .option("maxFilesPerTrigger", 2)
    .load(str(STREAM_SOURCE_DIR))
)

print(f"Est un streaming DataFrame : {stream_df.isStreaming}")
print(f"Colonnes : {stream_df.columns}")
# On ne peut pas appeler .count() ou .show() sur un streaming DataFrame

Est un streaming DataFrame : True
Colonnes : ['nom_station', 'capacite', 'velos_meca', 'velos_elec', 'velos_total', 'horodatage']


In [5]:
# Ajout des colonnes calculées -- exactement comme en batch
# Mêmes formules qu'au Jour 2 matin : bornettes_libres, taux_occupation,
# et un indicateur booléen "station vide" qui me servira pour les alertes
# plus bas (2.6).
stream_enrichi = (
    stream_df
    .withColumn("bornettes_libres", F.col("capacite") - F.col("velos_total"))
    .withColumn("taux_occupation", F.round(F.col("velos_total") / F.col("capacite"), 4))
    .withColumn("est_vide", F.col("velos_total") == 0)
)
print("Colonnes après enrichissement :", stream_enrichi.columns)

Colonnes après enrichissement : ['nom_station', 'capacite', 'velos_meca', 'velos_elec', 'velos_total', 'horodatage', 'bornettes_libres', 'taux_occupation', 'est_vide']


---
## 2.4 Première requête : sink console

La sortie `console` écrit les résultats dans le terminal Jupyter.
C'est utile pour le débogage -- jamais pour la production.


In [6]:
# Requête de streaming vers la console
# outputMode :
#   "append"  -- écrit uniquement les nouvelles lignes (défaut pour les flux sans agrégation)
#   "complete" -- réécrit l'intégralité du résultat à chaque batch
#   "update"  -- écrit uniquement les lignes qui ont changé
#
# La fonction start() lance le streaming
# (colonnes adaptées : nom_station remplace station_id/code_arr, cf. 2.3)
q_console = (
    stream_enrichi
    .select("nom_station", "horodatage", "velos_total", "taux_occupation", "est_vide")
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", "false")
    .option("numRows", 10)
    .trigger(processingTime="5 seconds")
    .start()
)

# La requête tourne en arrière-plan.
# awaitTermination() bloquerait indéfiniment -- on attend juste quelques batchs.
time.sleep(20)
print(f"Statut de la requête : {q_console.status}")
print(f"Batchs traités : {q_console.lastProgress['numInputRows'] if q_console.lastProgress else 'N/A'}")
q_console.stop()
print("Requête console arrêtée.")

Statut de la requête : {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Batchs traités : 0
Requête console arrêtée.


---
## 2.5 Agrégations sur fenêtres temporelles glissantes

L'agrégation sur des **fenêtres temporelles** est la requête streaming la plus
courante en pratique. Elle répond à des questions du type :
"Combien de vélos sont disponibles par arrondissement sur les 10 dernières minutes ?"

Spark propose deux types de fenêtres temporelles :

| Type | Définition | Exemple |
|------|-----------|---------|
| Fenêtre **basculante** | Fenêtres fixes, sans chevauchement | 0-10 min, 10-20 min... |
| Fenêtre **glissante** | Fenêtres qui se chevauchent | [0-10], [5-15], [10-20]... |

```
Fenêtre basculante (10 min)  : |--w1--|--w2--|--w3--|
Fenêtre glissante (10/5)   : |--w1--|          Taille=10, pas=5
                                  |--w2--|
                                       |--w3--|
```


In [7]:
from pyspark.sql.functions import window

# Agrégation glissante : disponibilité moyenne par station
# (je fais le regroupement par station plutôt que par arrondissement : ce
# dernier n'existe pas dans nos données, cf. adaptation en 2.3 -- le principe
# de la fenêtre glissante ne change pas).
# Fenêtre de 10 minutes, avançant toutes les 2 minutes.
# Le withWatermark est indispensable ici : sans lui, Spark garderait l'état
# de toutes les fenêtres indéfiniment en mémoire.
df_fenetre_arr = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .groupBy(
        window(F.col("horodatage"), "10 minutes", "2 minutes"),
        F.col("nom_station"),
    )
    .agg(
        F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"),
        F.count("*").alias("nb_observations"),
    )
    .withColumn("fenetre_debut", F.col("window.start"))
    .withColumn("fenetre_fin", F.col("window.end"))
    .drop("window")
)

print("Schéma de la requête fenêtrée :")
df_fenetre_arr.printSchema()

Schéma de la requête fenêtrée :
root
 |-- nom_station: string (nullable = true)
 |-- taux_moyen: double (nullable = true)
 |-- nb_observations: long (nullable = false)
 |-- fenetre_debut: timestamp (nullable = true)
 |-- fenetre_fin: timestamp (nullable = true)



In [8]:
# Écriture vers Delta Lake (sink delta) -- mode append
# outputMode="append" est le seul compatible avec un sink de fichiers pour
# une agrégation fenêtrée : Spark n'écrit une fenêtre qu'une fois qu'elle
# est "fermée" par le watermark.
path_fenetres       = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
checkpoint_fenetres = str(STREAM_CHECKPOINT / "fenetres_arrondissement")

q_fenetres = (
    df_fenetre_arr
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_fenetres)
    .trigger(processingTime="10 seconds")
    .queryName("q_fenetres_arrondissement")
    .start(path_fenetres)
)

print(f"Requête démarrée : {q_fenetres.name}")
print(f"ID               : {q_fenetres.id}")
print("En attente de 3 batchs...")
time.sleep(35)
print(f"Dernier progrès : {q_fenetres.lastProgress}")

Requête démarrée : q_fenetres_arrondissement
ID               : 1268bdc6-b310-44f9-a12c-166f7eac2a50
En attente de 3 batchs...
Dernier progrès : {'id': '1268bdc6-b310-44f9-a12c-166f7eac2a50', 'runId': '02f57a4c-3119-4a55-8c1c-a5dde8ec5d38', 'name': 'q_fenetres_arrondissement', 'timestamp': '2026-08-31T18:50:10.005Z', 'batchId': 2, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 29, 'triggerExecution': 34}, 'eventTime': {'watermark': '2026-08-31T18:33:13.000Z'}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[file:/home/jovyan/data/output/stream_input]', 'startOffset': {'logOffset': 1}, 'endOffset': {'logOffset': 1}, 'latestOffset': None, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0}], 'sink': {'description': 'DeltaSink[../data/output/delta/fenetres_arrondissement]', 'numOutputRows': -1}}


In [9]:
# Lecture des résultats accumulés dans Delta
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

try:
    # Je relis la table Delta alimentée en continu par q_fenetres et je trie
    # par début de fenêtre pour voir la chronologie des agrégats.
    df_lecture_fenetres = spark.read.format("delta").load(path_fenetres)
    df_lecture_fenetres.orderBy("fenetre_debut").show(30, truncate=False)
except Exception as e:
    print(f"Pas encore de données : {e}")
    print("Attendez encore quelques secondes et relancez cette cellule.")

+----------------------+----------+---------------+-------------------+-------------------+
|nom_station           |taux_moyen|nb_observations|fenetre_debut      |fenetre_fin        |
+----------------------+----------+---------------+-------------------+-------------------+
|Station Test Watermark|0.0       |10             |2026-08-31 18:20:00|2026-08-31 18:30:00|
|Station Test Watermark|0.0       |10             |2026-08-31 18:22:00|2026-08-31 18:32:00|
+----------------------+----------+---------------+-------------------+-------------------+



---
## 2.6 Alertes : détection de stations en rupture prolongée

L'objectif opérationnel est d'alerter l'équipe de maintenance quand une station
reste vide (zéro vélo disponible) pendant au moins **deux fenêtres consécutives**,
soit 20 minutes de rupture continue.

### Approche : `foreachBatch`

Pour une logique d'alerte complexe (état entre batchs, seuil consécutif),
`foreachBatch` est plus adapté que les agrégations pures. Il permet d'exécuter
une fonction Python arbitraire sur chaque micro-batch.


In [10]:
# Table d'alertes : schema
# Là encore, pas de station_id/code_arr disponibles (cf. 2.3) : j'identifie
# la station alertée par son nom.
schema_alertes = StructType([
    StructField("nom_station",    StringType(),    True),
    StructField("debut_rupture",  TimestampType(), True),
    StructField("fin_rupture",    TimestampType(), True),
    StructField("duree_min",      IntegerType(),   True),
    StructField("ts_alerte",      TimestampType(), True),
])

# Initialisation de la table Delta d'alertes (vide) -- juste pour fixer le
# schéma dès le départ ; traiter_batch_alertes viendra l'alimenter en append.
(
    spark.createDataFrame([], schema_alertes)
    .write
    .format("delta")
    .mode("overwrite")
    .save(str(DELTA_ALERTES))
)
print(f"Table d'alertes initialisée : {DELTA_ALERTES}")

Table d'alertes initialisée : ../data/output/delta/alertes


In [11]:
from delta.tables import DeltaTable
import datetime

# Mémoire inter-batchs : stations actuellement en rupture et depuis quand
etat_ruptures: dict[str, dict] = {}

def traiter_batch_alertes(batch_df, batch_id: int) -> None:
    """Fonction appelée par foreachBatch pour chaque micro-batch.

    Détecte les stations qui passent en rupture ou sortent de rupture.
    Écrit les alertes complètes dans la table Delta.

    Args:
        batch_df : DataFrame du micro-batch courant.
        batch_id : Identifiant séquentiel du batch.
    """
    global etat_ruptures

    # Rien à faire si le micro-batch est vide (ça arrive, notamment à cause
    # du watermark qui peut retenir des données).
    if batch_df.rdd.isEmpty():
        print(f"[batch {batch_id}] vide, rien à traiter")
        return

    # Pour chaque station présente dans ce batch : la fenêtre temporelle
    # couverte (ts_min/ts_max) et si TOUTES les observations du batch sont
    # à zéro vélo -- c'est le signal que j'utilise pour dire "en rupture sur
    # ce batch".
    etat_batch = (
        batch_df
        .groupBy("nom_station")
        .agg(
            F.min("horodatage").alias("ts_min"),
            F.max("horodatage").alias("ts_max"),
            F.sum(F.col("est_vide").cast("int")).alias("nb_vide"),
            F.count("*").alias("nb_total"),
        )
        .collect()  # un micro-batch tient en mémoire, agrégé par station c'est petit
    )

    alertes = []
    # Examiner les transitions :
    # - non vide -> vide : la station entre en rupture, je mémorise le début
    # - vide -> non vide : la station sort de rupture, je calcule la durée
    #   et je déclenche une alerte si elle dépasse 10 minutes
    for row in etat_batch:
        station = row["nom_station"]
        en_rupture_ce_batch = row["nb_vide"] == row["nb_total"]

        if station not in etat_ruptures:
            if en_rupture_ce_batch:
                etat_ruptures[station] = {"debut": row["ts_min"]}
        else:
            if not en_rupture_ce_batch:
                debut = etat_ruptures[station]["debut"]
                fin = row["ts_min"]
                duree_min = int((fin - debut).total_seconds() // 60)
                if duree_min > 10:
                    alertes.append({
                        "nom_station": station,
                        "debut_rupture": debut,
                        "fin_rupture": fin,
                        "duree_min": duree_min,
                        "ts_alerte": datetime.datetime.now(),
                    })
                del etat_ruptures[station]
            # sinon : toujours en rupture, le début mémorisé ne bouge pas

    # Si des alertes ont été enregistrées, les sauvegarder au format delta
    if alertes:
        (
            spark.createDataFrame(alertes, schema=schema_alertes)
            .write
            .format("delta")
            .mode("append")
            .save(str(DELTA_ALERTES))
        )
        print(f"[batch {batch_id}] {len(alertes)} alerte(s) écrite(s) dans {DELTA_ALERTES}")
    else:
        print(f"[batch {batch_id}] aucune alerte -- {len(etat_ruptures)} station(s) actuellement en rupture")

In [ ]:
# Lancement de la requête d'alerte
# - avec une tolérance de 5 minutes sur l'horodatage (cf. section suivante)
# - qui exécute la fonction `traiter_batch_alertes` pour chaque batch
# - définissant un `checkpoint` pour la sauvegarde de l'état en cas de panne
# - la requête s'appelle `q_alertes_rupture`
# - avec un intervalle de traitement de 10 secondes
checkpoint_alertes = str(STREAM_CHECKPOINT / "alertes_rupture")

q_alertes = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .writeStream
    .foreachBatch(traiter_batch_alertes)
    .option("checkpointLocation", checkpoint_alertes)
    .queryName("q_alertes_rupture")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Requête d'alertes démarrée. En attente de données...")
time.sleep(60)   # laisser tourner 1 minute pour accumuler des événements

# Lecture des alertes produites
df_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
print(f"\nAlertes enregistrées : {df_alertes.count()}")
df_alertes.orderBy(F.desc("ts_alerte")).show(20, truncate=False)

Requête d'alertes démarrée. En attente de données...

Alertes enregistrées : 0
+-----------+-------------+-----------+---------+---------+
|nom_station|debut_rupture|fin_rupture|duree_min|ts_alerte|
+-----------+-------------+-----------+---------+---------+
+-----------+-------------+-----------+---------+---------+



[batch 2] aucune alerte -- 1 station(s) actuellement en rupture
[batch 3] aucune alerte -- 1 station(s) actuellement en rupture


---
## 2.7 Données tardives et watermark

Dans un système distribué, certains événements arrivent avec du retard
(réseau saturé, capteur hors ligne temporairement...). Spark doit décider
combien de temps attendre avant de considérer une fenêtre comme fermée.

C'est le rôle du **watermark** : il définit le retard maximal toléré.

```
Watermark de 5 minutes :
  Si le timestamp max observé est 10h30, Spark considère que
  toutes les données avant 10h25 sont arrivées.
  Les fenêtres fermées avant 10h25 ne seront plus mises à jour.
```

### Illustration avec des données tardives simulées


In [13]:
import json, datetime

# Injection de données tardives :
# Écrire dans le répertoire de streaming un fichier avec des horodatages "dans le passé"
# Les dates sont dans le format ISO
# Le fichier est injecté dans le répertoire surveillé par Spark.
# Chaque ligne est homogène à une observation (ou snapshot).
def injecter_donnees_tardives(retard_minutes: int, nb_lignes: int = 20):
    """Écrit un fichier JSON avec des horodatages retardés.

    Args:
        retard_minutes : Retard à simuler (en minutes).
        nb_lignes      : Nombre de snapshots à générer.
    """
    # Je pars de "maintenant" et je recule de retard_minutes, exactement
    # comme le fait le simulateur pour recaler ses propres timestamps.
    maintenant = datetime.datetime.now(datetime.timezone.utc)
    ts_retarde = maintenant - datetime.timedelta(minutes=retard_minutes)

    payload = [
        {
            "nom_station": "Station Test Watermark",
            "capacite": 20,
            "velos_meca": 0,
            "velos_elec": 0,
            "velos_total": 0,
            "horodatage": (ts_retarde + datetime.timedelta(seconds=i)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        }
        for i in range(nb_lignes)
    ]

    nom_fichier = STREAM_SOURCE_DIR / f"tardif_{retard_minutes}min_{int(time.time())}.json"
    nom_tmp     = STREAM_SOURCE_DIR / f".tmp_{nom_fichier.name}"
    with open(nom_tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    nom_tmp.rename(nom_fichier)  # écriture atomique, comme dans le simulateur

    print(f"[injection] {nom_fichier.name} -- retard={retard_minutes} min -- {nb_lignes} lignes")

# Scénario 1 : retard de 3 min -- dans le watermark (5 min) -> données prises en compte
injecter_donnees_tardives(retard_minutes=3, nb_lignes=10)
time.sleep(15)

# Scénario 2 : retard de 12 min -- hors watermark -> données ignorées par les agrégations
injecter_donnees_tardives(retard_minutes=12, nb_lignes=10)
time.sleep(15)

print("\nStatut des requêtes actives :")
for q in spark.streams.active:
    prog = q.lastProgress
    print(f"  {q.name:<30} -- inputRows : {prog['numInputRows'] if prog else 'N/A'}")

[injection] tardif_3min_1788202295.json -- retard=3 min -- 10 lignes
[injection] tardif_12min_1788202310.json -- retard=12 min -- 10 lignes

Statut des requêtes actives :
  q_alertes_rupture              -- inputRows : 20
  q_fenetres_arrondissement      -- inputRows : 10


In [14]:
# [EXERCICE]
# Créez une nouvelle requête de streaming qui calcule,
# pour chaque arrondissement et sur une fenêtre basculante de 15 minutes,
# le MAXIMUM du taux_occupation observé.
# Écrivez le résultat en mode "update" vers la console.
#
# Rappel : fenêtre basculante = window(col, "15 minutes")  (sans 3e argument)
# ──────────────────────────────────────────────────────────────────────────

# [CORRECTION] -- ma solution
# Comme pour 2.5, je groupe par station (pas d'arrondissement dans nos
# données). Fenêtre basculante 15 min -> j'appelle window() avec un seul
# argument de durée, donc pas de chevauchement entre fenêtres consécutives.
q_exercice = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .groupBy(window(F.col("horodatage"), "15 minutes"), F.col("nom_station"))
    .agg(F.round(F.max("taux_occupation"), 4).alias("taux_max"))
    .writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", "false")
    .trigger(processingTime="15 seconds")
    .start()
)

time.sleep(20)
q_exercice.stop()
print("Requête d'exercice arrêtée.")

Requête d'exercice arrêtée.


---
## 2.8 Arrêt propre des requêtes et bilan

En production, les requêtes de streaming tournent indéfiniment.
Dans un contexte de cours, il faut les arrêter proprement pour libérer les ressources.


In [15]:
# Arrêt de toutes les requêtes actives
actives = spark.streams.active
print(f"Requêtes actives à arrêter : {[q.name for q in actives]}")

# J'arrête chaque requête proprement avec .stop() plutôt que de couper le
# kernel : ça libère les threads de streaming et ferme les checkpoints
# correctement.
for q in actives:
    q.stop()
    print(f"  -- {q.name} arrêtée")

print("\nToutes les requêtes de streaming sont arrêtées.")

Requêtes actives à arrêter : ['q_alertes_rupture', 'q_fenetres_arrondissement']
  -- q_alertes_rupture arrêtée
  -- q_fenetres_arrondissement arrêtée

Toutes les requêtes de streaming sont arrêtées.


In [16]:
# Lecture finale des résultats accumulés
print("=== Résultats finaux ===")

print("\n-- Fenêtres arrondissement --")
try:
    df_fin_fenetres = spark.read.format("delta").load(
        str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
    )
    print(f"  {df_fin_fenetres.count()} fenêtres enregistrées")
    df_fin_fenetres.orderBy(F.desc("fenetre_debut")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucun résultat : {e}")

print("\n-- Alertes rupture --")
try:
    df_fin_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
    print(f"  {df_fin_alertes.count()} alertes enregistrées")
    df_fin_alertes.orderBy(F.desc("ts_alerte")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucune alerte : {e}")


=== Résultats finaux ===

-- Fenêtres arrondissement --
  7 fenêtres enregistrées
+----------------------+----------+---------------+-------------------+-------------------+
|nom_station           |taux_moyen|nb_observations|fenetre_debut      |fenetre_fin        |
+----------------------+----------+---------------+-------------------+-------------------+
|Station Test Watermark|0.0       |10             |2026-08-31 18:32:00|2026-08-31 18:42:00|
|Station Test Watermark|0.0       |10             |2026-08-31 18:30:00|2026-08-31 18:40:00|
|Station Test Watermark|0.0       |10             |2026-08-31 18:28:00|2026-08-31 18:38:00|
|Station Test Watermark|0.0       |10             |2026-08-31 18:26:00|2026-08-31 18:36:00|
|Station Test Watermark|0.0       |10             |2026-08-31 18:24:00|2026-08-31 18:34:00|
|Station Test Watermark|0.0       |10             |2026-08-31 18:22:00|2026-08-31 18:32:00|
|Station Test Watermark|0.0       |10             |2026-08-31 18:20:00|2026-08-31 18:30:00

---
## Bilan du Jour 2

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Vues temporaires et SQL de base | Spark SQL | `createOrReplaceTempView`, SQL natif |
| Questions métier analytiques | Spark SQL | `LEFT ANTI JOIN`, `CASE WHEN`, sous-requêtes |
| Fenêtrage analytique | Spark SQL / DataFrame | `OVER`, `LAG`, `LEAD`, `ROW_NUMBER`, `AVG OVER` |
| Écriture Delta Lake | Delta | `format("delta")`, partitionnement |
| Time-travel | Delta | `versionAsOf`, historique des opérations |
| Mise à jour incrémentale | Delta | `MERGE INTO`, `whenMatchedUpdateAll` |
| Source de streaming | Structured Streaming | `readStream`, schéma explicite, file source |
| Sink console | Structured Streaming | débogage, `outputMode("append")` |
| Fenêtres glissantes | Structured Streaming | `window()`, basculante vs glissante |
| Sink Delta | Structured Streaming | écriture transactionnelle en flux |
| Alertes avec `foreachBatch` | Structured Streaming | logique inter-batchs, état partagé |
| Watermark et late data | Structured Streaming | tolérance, fermeture de fenêtres |

### Points d'attention

- En `outputMode("append")`, Spark n'écrit que des lignes **nouvelles et définitives**.
  Pour les agrégations fenêtrées, cela implique un watermark : Spark attend que la fenêtre
  soit fermée avant d'émettre son résultat.
- `foreachBatch` est puissant mais introduit un état mutable (`etat_ruptures`) qui
  n'est **pas persisté dans le checkpoint**. En cas de redémarrage, l'état est perdu.
  Pour un système de production, il faudrait utiliser `mapGroupsWithState` ou
  stocker l'état dans Delta Lake.
- Le checkpoint est **obligatoire** pour toute requête qui ne doit pas repartir de zéro
  après un redémarrage.

### Pour demain (Jour 3)

La table Delta `disponibilite` (batch) et les résultats des fenêtres glissantes
(streaming) serviront de données d'entrée pour le Jour 3 : construction des features,
entraînement d'un modèle de régression avec MLlib, clustering des stations avec K-Means
et suivi des expériences avec MLflow.


In [17]:
spark.stop()
print("SparkSession arrêtée. À demain !")


SparkSession arrêtée. À demain !
